# Signal taxonomy — what each signal is and where it is valid
_Reference card for the descriptive layer: every mart signal's class, validity, and per-position liveness — the one place the other notebooks' signal choices are justified._

**Sections:** (a) signal reference card · (b) position relevance map

---

## Setup
> Whole season, `minutes > 0` participation — the relevance map needs the data; the taxonomy table is pure domain.

In [1]:
import pandas as pd
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from domain.signal_layers import SIGNAL_LAYER_MAPPING
from research.kernels.descriptive.relevance import (
    compute_relevance,
    formula_input_signals,
    leading_indicator_signals,
    signal_class,
    POSITIONS,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 170)

try:
    _r = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _r = load_mart()

mart = _r.mart
df = mart[mart["gw"].between(1, _r.data_cutoff_gw)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()

formula = formula_input_signals()
leading = leading_indicator_signals()
print(f"Study range: GW 1 - {_r.data_cutoff_gw} · minutes > 0 · n = {len(df):,} player-gameweeks")
print(f"{len(formula)} formula inputs · {len(leading)} leading indicators")

Study range: GW 1 - 38 · minutes > 0 · n = 11,361 player-gameweeks
12 formula inputs · 11 leading indicators


## (a) Signal reference card
> What class is each signal, and what is it valid for?

Formula inputs (`layer_role` in `TAUTOLOGICAL_LAYER_ROLES`) are valid for distribution / frequency / decomposition only — their same-gameweek association with `total_points` is mechanically set by the scoring formula. Leading indicators (`feature_candidate_eligible`, not tautological) are the only signals valid for association and X-vs-X correlation.

In [2]:
VALID_FOR = {
    "formula_input": "distribution / frequency / decomposition",
    "leading_indicator": "association + X-vs-X correlation",
}
rows = []
for sig in sorted(formula | leading):
    meta = SIGNAL_LAYER_MAPPING[sig]
    cls = signal_class(sig)
    rows.append({
        "signal": sig,
        "signal_class": cls,
        "signal_layer": meta["signal_layer"],
        "layer_role": meta["layer_role"],
        "feature_candidate_eligible": meta["feature_candidate_eligible"],
        "valid_for": VALID_FOR[cls],
        "interpretation_caveat": meta["interpretation_caveat"],
    })
taxonomy = (
    pd.DataFrame(rows)
    .sort_values(["signal_class", "signal"])
    .reset_index(drop=True)
)
display(taxonomy)

,signal,signal_class,signal_layer,layer_role,feature_candidate_eligible,valid_for,interpretation_caveat
0,assists,formula_input,performance,creation_event,True,distribution / frequency / decomposition,sparse event caveats apply
1,bonus,formula_input,performance,points_component,True,distribution / frequency / decomposition,scoring outcome component; avoid target leakag...
2,bps,formula_input,performance,contribution_index,True,distribution / frequency / decomposition,"descriptive contribution index, not causal"
3,clean_sheets,formula_input,defensive_context,defensive_points_context,False,distribution / frequency / decomposition,team/role dependent scoring context
4,goals_conceded,formula_input,defensive_context,defensive_outcome_context,False,distribution / frequency / decomposition,"team defensive outcome, not player quality"
5,goals_scored,formula_input,performance,scoring_event,True,distribution / frequency / decomposition,sparse event caveats apply
6,own_goals,formula_input,discipline,negative_event,True,distribution / frequency / decomposition,sparse/low-frequency caveats apply
7,penalties_missed,formula_input,discipline,negative_event,True,distribution / frequency / decomposition,sparse/low-frequency caveats apply
8,penalties_saved,formula_input,defensive_context,goalkeeper_action,True,distribution / frequency / decomposition,GK-specific; sparse event caveats apply
9,red_cards,formula_input,discipline,negative_event,True,distribution / frequency / decomposition,sparse/low-frequency caveats apply


## (b) Position relevance map
> Which signals are alive (carry mass and variance) at each position, and which are structural zeros?

`structural_zero` marks a leading indicator absent at that position (e.g. attacking xG signals at GK); `formula_input_dead` marks a formula input too sparse to decompose there. Both come from the shared relevance kernel (zero-mass >= 93% or near-zero variance) — `leading_alive` is the only class carried forward to association analysis.

In [3]:
relevance = compute_relevance(df, group_cols=["position"])
rel_pivot = relevance.pivot(index="signal", columns="position", values="relevance")[list(POSITIONS)]
# Order rows by class then signal so formula inputs and leading indicators group visually.
rel_pivot = rel_pivot.reindex(taxonomy.set_index("signal").index)
print("leading_alive       = testable for association")
print("structural_zero     = leading indicator absent at this position")
print("formula_input       = valid for distribution / decomposition, not association with Y")
print("formula_input_dead  = formula input AND too sparse at this position")
display(rel_pivot)

leading_alive       = testable for association
structural_zero     = leading indicator absent at this position
formula_input       = valid for distribution / decomposition, not association with Y
formula_input_dead  = formula input AND too sparse at this position


position,GK,DEF,MID,FWD
signal,,,,
assists,formula_input_dead,formula_input_dead,formula_input,formula_input_dead
bonus,formula_input,formula_input,formula_input,formula_input
bps,formula_input,formula_input,formula_input,formula_input
clean_sheets,formula_input,formula_input,formula_input,formula_input
goals_conceded,formula_input,formula_input,formula_input,formula_input
goals_scored,formula_input_dead,formula_input_dead,formula_input,formula_input
own_goals,formula_input_dead,formula_input_dead,formula_input_dead,formula_input_dead
penalties_missed,formula_input_dead,formula_input_dead,formula_input_dead,formula_input_dead
penalties_saved,formula_input_dead,formula_input_dead,formula_input_dead,formula_input_dead
